
# Mars Explorer — RAG-Powered Document Assistant

**Level 2 Summer Training | Graduation Project**  
**Track:** Core Track — Text-based RAG Assistant  
**Environment:** Google Colab  
**LLM:** Ollama `llama3.2:3b`  
**Embeddings:** `sentence-transformers/all-MiniLM-L6-v2`  
**Vector Store:** FAISS (persisted to disk)  
**Domain:** NASA Mars exploration

## Project Goal
Build and evaluate a complete Retrieval-Augmented Generation (RAG) pipeline using authoritative NASA Mars documents. The pipeline downloads the data, cleans it, chunks it, creates embeddings, stores them in a persisted FAISS vector index, retrieves relevant chunks, builds a grounded prompt, calls a local Ollama LLM, returns cited answers, and evaluates the system on 10 test questions.

> **Colab note:** This notebook installs Ollama inside the Colab Linux runtime and launches the Ollama server locally in that runtime. Run cells from top to bottom.


## 0. Install Dependencies

In [1]:

!pip -q install sentence-transformers faiss-cpu beautifulsoup4 requests pandas numpy tqdm

print("Python dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.3 MB/s eta 0:00:00
Python dependencies installed.



### Install Ollama
The assignment requires a **local Ollama LLM**. This cell installs Ollama in the current Colab runtime.


In [2]:
import os
import shutil
import subprocess
import requests
from pathlib import Path

print("Installing Ollama in Colab...")

# Install zstd so Colab can extract .tar.zst
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd"],
    check=True
)

# Use current Ollama Linux AMD64 release asset
OLLAMA_URL = (
    "https://github.com/ollama/ollama/releases/latest/download/"
    "ollama-linux-amd64.tar.zst"
)

archive_path = "/content/ollama-linux-amd64.tar.zst"

r = requests.get(OLLAMA_URL, stream=True, timeout=300)
r.raise_for_status()

with open(archive_path, "wb") as f:
    for chunk in r.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

print("Download finished.")

# Extract into /usr
subprocess.run(
    [
        "tar",
        "--use-compress-program=unzstd",
        "-xf",
        archive_path,
        "-C",
        "/usr"
    ],
    check=True
)

ollama_path = shutil.which("ollama")

print("Ollama path:", ollama_path)

if ollama_path is None:
    raise RuntimeError("Ollama installation failed.")

print("Ollama installed successfully.")

Installing Ollama in Colab...
Download finished.
Ollama path: /usr/bin/ollama
Ollama installed successfully.


### Start Ollama Server and Pull the Model

In [3]:
import subprocess
import time
import requests

def ollama_is_running():
    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )
        return response.status_code == 200
    except Exception:
        return False


# Start Ollama server if it is not already running
if not ollama_is_running():
    print("Starting Ollama server...")

    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    for _ in range(30):
        if ollama_is_running():
            break
        time.sleep(1)


# Check server
if not ollama_is_running():
    raise RuntimeError("Ollama server failed to start.")

print("Ollama server is running.")


# Pull model
MODEL_NAME = "llama3.2:3b"

print("Downloading model:", MODEL_NAME)

subprocess.run(
    ["ollama", "pull", MODEL_NAME],
    check=True
)

print("Model ready:", MODEL_NAME)

Starting Ollama server...
Ollama server is running.
Model ready: llama3.2:3b



## 1. Domain & Data Collection

The project uses official NASA Science pages as the source corpus. Each page is downloaded and saved locally as a `.txt` document so the RAG system works with a real document collection.

### Data Sources
1. Mars Facts — https://science.nasa.gov/mars/facts/
2. Perseverance Rover — https://science.nasa.gov/mission/mars-2020-perseverance/
3. Perseverance Science — https://science.nasa.gov/mission/mars-2020-perseverance/science/
4. Curiosity Rover — https://science.nasa.gov/mission/msl-curiosity/
5. Curiosity Science — https://science.nasa.gov/mission/msl-curiosity/science/
6. InSight Lander — https://science.nasa.gov/mission/insight/
7. InSight Science — https://science.nasa.gov/mission/insight/science/
8. Ingenuity Mars Helicopter — https://science.nasa.gov/mission/mars-2020-perseverance/ingenuity-mars-helicopter/

All sources are from **NASA Science**.


In [4]:

from pathlib import Path
import requests, re, json
from bs4 import BeautifulSoup
import pandas as pd

PROJECT_DIR = Path("/content/mars_rag_project")
DATA_DIR = PROJECT_DIR / "data" / "documents"
VECTOR_DIR = PROJECT_DIR / "data" / "vector_store"
EXPORT_DIR = PROJECT_DIR / "exports"

for d in [DATA_DIR, VECTOR_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_URLS = {
    "mars_facts": "https://science.nasa.gov/mars/facts/",
    "perseverance": "https://science.nasa.gov/mission/mars-2020-perseverance/",
    "perseverance_science": "https://science.nasa.gov/mission/mars-2020-perseverance/science/",
    "curiosity": "https://science.nasa.gov/mission/msl-curiosity/",
    "curiosity_science": "https://science.nasa.gov/mission/msl-curiosity/science/",
    "insight": "https://science.nasa.gov/mission/insight/",
    "insight_science": "https://science.nasa.gov/mission/insight/science/",
    "ingenuity": "https://science.nasa.gov/mission/mars-2020-perseverance/ingenuity-mars-helicopter/",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Educational-RAG-Project/1.0)"
}

def clean_web_text(text):
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()

download_report = []

for name, url in SOURCE_URLS.items():
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg", "form", "nav", "footer"]):
        tag.decompose()

    main = soup.find("main")
    if main is None:
        main = soup.body

    text = clean_web_text(main.get_text("\n", strip=True))
    output = f"TITLE: {name.replace('_', ' ').title()}\nSOURCE_URL: {url}\n\n{text}"

    file_path = DATA_DIR / f"{name}.txt"
    file_path.write_text(output, encoding="utf-8")

    download_report.append({
        "document": name,
        "url": url,
        "characters": len(output),
        "saved_as": str(file_path)
    })

download_df = pd.DataFrame(download_report)
display(download_df)

if (download_df["characters"] < 500).any():
    raise RuntimeError("One or more NASA pages produced too little text. Check the URLs/network.")

print(f"\nDownloaded {len(download_df)} NASA documents.")


,document,url,characters,saved_as
0,mars_facts,https://science.nasa.gov/mars/facts/,9593,/content/mars_rag_project/data/documents/mars_...
1,perseverance,https://science.nasa.gov/mission/mars-2020-per...,9374,/content/mars_rag_project/data/documents/perse...
2,perseverance_science,https://science.nasa.gov/mission/mars-2020-per...,3865,/content/mars_rag_project/data/documents/perse...
3,curiosity,https://science.nasa.gov/mission/msl-curiosity/,6275,/content/mars_rag_project/data/documents/curio...
4,curiosity_science,https://science.nasa.gov/mission/msl-curiosity...,5064,/content/mars_rag_project/data/documents/curio...
5,insight,https://science.nasa.gov/mission/insight/,5442,/content/mars_rag_project/data/documents/insig...
6,insight_science,https://science.nasa.gov/mission/insight/science/,3420,/content/mars_rag_project/data/documents/insig...
7,ingenuity,https://science.nasa.gov/mission/mars-2020-per...,8065,/content/mars_rag_project/data/documents/ingen...



Downloaded 8 NASA documents.



## 2.1 Load & Inspect

This section verifies the downloaded corpus before building the RAG pipeline.

The corpus contains HTML-derived text saved as UTF-8 `.txt` files. Because the sources are normal text webpages, OCR is not required.


In [5]:

documents = []
failures = []

for path in sorted(DATA_DIR.glob("*.txt")):
    try:
        text = path.read_text(encoding="utf-8").strip()
        source_url_match = re.search(r"SOURCE_URL:\s*(.+)", text)
        source_url = source_url_match.group(1).strip() if source_url_match else ""

        documents.append({
            "document_id": path.stem,
            "filename": path.name,
            "source_url": source_url,
            "text": text,
            "characters": len(text),
            "words": len(text.split())
        })
    except Exception as e:
        failures.append({"file": path.name, "error": str(e)})

docs_df = pd.DataFrame(documents)
display(docs_df[["document_id", "filename", "characters", "words", "source_url"]])

print("Number of documents:", len(documents))
print("Formats: .txt")
print("Files needing OCR: 0")
print("Parse failures:", failures if failures else "None")


,document_id,filename,characters,words,source_url
0,curiosity,curiosity.txt,6275,1019,https://science.nasa.gov/mission/msl-curiosity/
1,curiosity_science,curiosity_science.txt,5064,743,https://science.nasa.gov/mission/msl-curiosity...
2,ingenuity,ingenuity.txt,8065,1226,https://science.nasa.gov/mission/mars-2020-per...
3,insight,insight.txt,5442,862,https://science.nasa.gov/mission/insight/
4,insight_science,insight_science.txt,3420,547,https://science.nasa.gov/mission/insight/science/
5,mars_facts,mars_facts.txt,9593,1627,https://science.nasa.gov/mars/facts/
6,perseverance,perseverance.txt,9374,1453,https://science.nasa.gov/mission/mars-2020-per...
7,perseverance_science,perseverance_science.txt,3865,577,https://science.nasa.gov/mission/mars-2020-per...


Number of documents: 8
Formats: .txt
Files needing OCR: 0
Parse failures: None



## 2.2 Chunking Strategy

A **fixed-size word chunking strategy with overlap** is used.

- **Chunk size:** 220 words
- **Overlap:** 40 words

### Justification
A chunk of roughly 220 words is large enough to preserve a complete idea or group of related facts while remaining focused enough for semantic retrieval. A 40-word overlap helps preserve context when an important sentence lies close to a chunk boundary.


In [6]:

CHUNK_SIZE = 220
CHUNK_OVERLAP = 40

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks

chunks = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"])

    for chunk_index, chunk in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f'{doc["document_id"]}_chunk_{chunk_index:03d}',
            "document_id": doc["document_id"],
            "filename": doc["filename"],
            "source_url": doc["source_url"],
            "chunk_index": chunk_index,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))
display(chunks_df.head())


Total chunks: 45


,chunk_id,document_id,filename,source_url,chunk_index,text
0,curiosity_chunk_000,curiosity,curiosity.txt,https://science.nasa.gov/mission/msl-curiosity/,0,TITLE: Curiosity SOURCE_URL: https://science.n...
1,curiosity_chunk_001,curiosity,curiosity.txt,https://science.nasa.gov/mission/msl-curiosity/,1,it marked an elevation gain of 1 kilometer in ...
2,curiosity_chunk_002,curiosity,curiosity.txt,https://science.nasa.gov/mission/msl-curiosity/,2,Mars lab inside the belly of NASA’s Curiosity ...
3,curiosity_chunk_003,curiosity,curiosity.txt,https://science.nasa.gov/mission/msl-curiosity/,3,end of its robotic arm. This grid shows all 42...
4,curiosity_chunk_004,curiosity,curiosity.txt,https://science.nasa.gov/mission/msl-curiosity/,4,back by Curiosity from its explorations on Mar...



## 2.3 Embeddings & Persistent FAISS Vector Store

The embedding model is `sentence-transformers/all-MiniLM-L6-v2`.

FAISS is used as the vector database. The index and chunk metadata are saved to disk so the backend can load them without rebuilding embeddings at request time.


In [7]:

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import json

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)

INDEX_PATH = VECTOR_DIR / "mars_faiss.index"
METADATA_PATH = VECTOR_DIR / "chunks.json"
CONFIG_PATH = VECTOR_DIR / "config.json"

faiss.write_index(faiss_index, str(INDEX_PATH))
chunks_df.to_json(METADATA_PATH, orient="records", force_ascii=False, indent=2)

config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunk_size_words": CHUNK_SIZE,
    "chunk_overlap_words": CHUNK_OVERLAP,
    "vector_store": "FAISS IndexFlatIP",
    "number_of_documents": len(documents),
    "number_of_chunks": len(chunks_df),
    "embedding_dimension": int(dimension),
    "ollama_model": MODEL_NAME
}

CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("Vector store persisted:")
print(" -", INDEX_PATH)
print(" -", METADATA_PATH)
print(" -", CONFIG_PATH)
print("\nFAISS vectors:", faiss_index.ntotal)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Vector store persisted:
 - /content/mars_rag_project/data/vector_store/mars_faiss.index
 - /content/mars_rag_project/data/vector_store/chunks.json
 - /content/mars_rag_project/data/vector_store/config.json

FAISS vectors: 45


## 2.4 Retrieval & Prompting

In [8]:

TOP_K = 4

def retrieve(question, top_k=TOP_K):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, top_k)

    results = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        row = chunks_df.iloc[int(idx)]

        results.append({
            "rank": rank,
            "score": float(score),
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "filename": row["filename"],
            "source_url": row["source_url"],
            "text": row["text"]
        })

    return results

sample_question = "What was the main objective of NASA's InSight mission?"
sample_results = retrieve(sample_question)

display(pd.DataFrame(sample_results)[
    ["rank", "score", "document_id", "chunk_id", "source_url"]
])


,rank,score,document_id,chunk_id,source_url
0,1,0.521209,insight_science,insight_science_chunk_001,https://science.nasa.gov/mission/insight/science/
1,2,0.509570,insight,insight_chunk_001,https://science.nasa.gov/mission/insight/
2,3,0.494748,insight,insight_chunk_000,https://science.nasa.gov/mission/insight/
3,4,0.480607,insight,insight_chunk_002,https://science.nasa.gov/mission/insight/


In [9]:

SYSTEM_INSTRUCTION = """You are Mars Explorer, a grounded RAG assistant.

Rules:
1. Answer ONLY from the retrieved context.
2. Do not use outside knowledge.
3. If the answer is not supported by the context, say:
   "I don't know based on the provided NASA documents."
4. Cite factual statements with [Source 1], [Source 2], etc.
5. Keep the answer clear and concise.
"""

def build_prompt(question, retrieved_chunks):
    context_parts = []

    for i, item in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Source {i}]\n"
            f"Document: {item['filename']}\n"
            f"URL: {item['source_url']}\n"
            f"Content:\n{item['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = (
        SYSTEM_INSTRUCTION
        + "\n\nRETRIEVED CONTEXT\n-----------------\n"
        + context
        + "\n\nUSER QUESTION\n-------------\n"
        + question
        + "\n\nANSWER\n------\n"
    )

    return prompt


### Ollama Generation

In [10]:

import requests

OLLAMA_API = "http://127.0.0.1:11434/api/generate"

def generate_with_ollama(prompt, model=MODEL_NAME, temperature=0.1):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    response = requests.post(OLLAMA_API, json=payload, timeout=300)
    response.raise_for_status()
    return response.json()["response"].strip()

def ask_rag(question, top_k=TOP_K, show_context=False):
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)
    answer = generate_with_ollama(prompt)

    result = {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "source_number": i,
                "document": item["filename"],
                "chunk_id": item["chunk_id"],
                "url": item["source_url"],
                "score": round(item["score"], 4)
            }
            for i, item in enumerate(retrieved, start=1)
        ]
    }

    print("QUESTION:")
    print(question)
    print("\nANSWER:")
    print(answer)
    print("\nRETRIEVED SOURCES:")

    for source in result["sources"]:
        print(
            f"[Source {source['source_number']}] "
            f"{source['document']} | {source['chunk_id']} | "
            f"score={source['score']}"
        )
        print(source["url"])

    if show_context:
        print("\nCONTEXT:")
        for item in retrieved:
            print("\n---", item["chunk_id"], "---")
            print(item["text"][:1000])

    return result


### Quick End-to-End Test

In [11]:

demo_result = ask_rag(
    "What was the main objective of the Curiosity rover?"
)


QUESTION:
What was the main objective of the Curiosity rover?

ANSWER:
According to [Source 3], the main objective of the Curiosity rover was to determine if Mars was ever able to support microbial life.

RETRIEVED SOURCES:
[Source 1] curiosity.txt | curiosity_chunk_003 | score=0.5988
https://science.nasa.gov/mission/msl-curiosity/
[Source 2] curiosity.txt | curiosity_chunk_002 | score=0.5562
https://science.nasa.gov/mission/msl-curiosity/
[Source 3] curiosity.txt | curiosity_chunk_000 | score=0.5386
https://science.nasa.gov/mission/msl-curiosity/
[Source 4] perseverance.txt | perseverance_chunk_007 | score=0.5353
https://science.nasa.gov/mission/mars-2020-perseverance/


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



## Retrieval Testing — 10 Sample Questions

The assignment requires retrieval to be tested against at least 10 questions. The questions below cover multiple documents rather than testing only one topic.


In [14]:

TEST_QUESTIONS = [
    {"question": "How long is a day on Mars?", "expected_document": "mars_facts"},
    {"question": "Why does Mars look red?", "expected_document": "mars_facts"},
    {"question": "What evidence suggests Mars had water in the past?", "expected_document": "mars_facts"},
    {"question": "What is Perseverance searching for on Mars?", "expected_document": "perseverance"},
    {"question": "Why is Jezero Crater scientifically important to Perseverance?", "expected_document": "perseverance_science"},
    {"question": "What was the main objective of the Curiosity rover?", "expected_document": "curiosity"},
    {"question": "What is Curiosity investigating at Gale Crater?", "expected_document": "curiosity_science"},
    {"question": "What part of Mars did InSight study?", "expected_document": "insight"},
    {"question": "What did InSight learn using seismic measurements?", "expected_document": "insight_science"},
    {"question": "What historic achievement did the Ingenuity helicopter demonstrate?", "expected_document": "ingenuity"},
]

retrieval_rows = []

for item in TEST_QUESTIONS:
    retrieved = retrieve(item["question"], top_k=TOP_K)
    retrieved_docs = [r["document_id"] for r in retrieved]

    retrieval_rows.append({
        "question": item["question"],
        "expected_document": item["expected_document"],
        "top_1_document": retrieved_docs[0],
        "retrieved_documents": ", ".join(retrieved_docs),
        "expected_found_in_top_k": item["expected_document"] in retrieved_docs
    })

retrieval_eval_df = pd.DataFrame(retrieval_rows)
display(retrieval_eval_df)

retrieval_accuracy = retrieval_eval_df["expected_found_in_top_k"].mean()
print(f"Expected source found in Top-{TOP_K}: {retrieval_accuracy:.0%}")


,question,expected_document,top_1_document,retrieved_documents,expected_found_in_top_k
0,How long is a day on Mars?,mars_facts,mars_facts,"mars_facts, mars_facts, mars_facts, mars_facts",True
1,Why does Mars look red?,mars_facts,mars_facts,"mars_facts, mars_facts, mars_facts, mars_facts",True
2,What evidence suggests Mars had water in the p...,mars_facts,mars_facts,"mars_facts, perseverance, mars_facts, mars_facts",True
3,What is Perseverance searching for on Mars?,perseverance,perseverance_science,"perseverance_science, perseverance, perseveran...",True
4,Why is Jezero Crater scientifically important ...,perseverance_science,perseverance,"perseverance, perseverance, perseverance, pers...",True
5,What was the main objective of the Curiosity r...,curiosity,curiosity,"curiosity, curiosity, curiosity, perseverance",True
6,What is Curiosity investigating at Gale Crater?,curiosity_science,curiosity,"curiosity, curiosity_science, curiosity_scienc...",True
7,What part of Mars did InSight study?,insight,insight_science,"insight_science, insight, insight_science, ins...",True
8,What did InSight learn using seismic measureme...,insight_science,insight_science,"insight_science, insight_science, insight, ins...",True
9,What historic achievement did the Ingenuity he...,ingenuity,ingenuity,"ingenuity, ingenuity, perseverance, ingenuity",True


Expected source found in Top-4: 100%



## 2.6 RAG Evaluation

For each test question, the notebook records:

- retrieved source
- generated answer
- whether the expected document was retrieved
- whether the answer contains citation markers
- a final automatic pass indicator

This gives a transparent, reproducible evaluation table. The final answers should also be reviewed manually before submission because automatic metrics cannot fully judge factual correctness.


In [15]:

import re
from tqdm.auto import tqdm

evaluation_rows = []

for item in tqdm(TEST_QUESTIONS, desc="Running RAG evaluation"):
    question = item["question"]
    expected_document = item["expected_document"]

    retrieved = retrieve(question, top_k=TOP_K)
    prompt = build_prompt(question, retrieved)
    answer = generate_with_ollama(prompt)

    retrieved_docs = [r["document_id"] for r in retrieved]
    source_retrieved = expected_document in retrieved_docs
    has_citation = bool(re.search(r"\[Source\s+\d+\]", answer, flags=re.I))

    automatic_correct = bool(source_retrieved and has_citation)

    evaluation_rows.append({
        "question": question,
        "expected_source": expected_document,
        "retrieved_source_top1": retrieved[0]["document_id"],
        "retrieved_source_top_k": ", ".join(retrieved_docs),
        "answer": answer,
        "expected_source_retrieved": source_retrieved,
        "citation_present": has_citation,
        "automatic_correct": automatic_correct
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(
    evaluation_df[
        ["question", "retrieved_source_top1", "answer", "automatic_correct"]
    ]
)

print(
    "Automatic pass rate:",
    f"{evaluation_df['automatic_correct'].mean():.0%}"
)


Running RAG evaluation:   0%|          | 0/10 [00:00<?, ?it/s]

,question,retrieved_source_top1,answer,automatic_correct
0,How long is a day on Mars?,mars_facts,"According to the retrieved context, a day on M...",False
1,Why does Mars look red?,mars_facts,"According to [Source 1], Mars looks red due to...",True
2,What evidence suggests Mars had water in the p...,mars_facts,"According to [Source 1] and [Source 2], eviden...",True
3,What is Perseverance searching for on Mars?,perseverance_science,"According to [Source 1] and [Source 2], Persev...",True
4,Why is Jezero Crater scientifically important ...,perseverance,"According to [Source 1] and [Source 4], Jezero...",True
5,What was the main objective of the Curiosity r...,curiosity,"According to [Source 3], the main objective of...",True
6,What is Curiosity investigating at Gale Crater?,curiosity,"According to [Source 1] and [Source 2], Curios...",True
7,What part of Mars did InSight study?,insight_science,"According to [Source 1] and [Source 4], InSigh...",True
8,What did InSight learn using seismic measureme...,insight_science,I don't know based on the provided NASA docume...,False
9,What historic achievement did the Ingenuity he...,ingenuity,The Ingenuity helicopter demonstrated the firs...,False


Automatic pass rate: 70%



### Failure Case Analysis

Use the table above to inspect failures. Typical failure modes in this system are:

1. **Semantically similar mission pages:** A question about a rover may retrieve its overview page instead of its science page because both discuss the same mission.
2. **Chunk boundary effects:** A fact may be split across neighboring chunks. The 40-word overlap reduces this problem.
3. **General questions:** Broad Mars questions can match several chunks with similar scores.
4. **Generation grounding:** The prompt explicitly instructs the LLM to answer only from retrieved context and to say it does not know when the context is insufficient.
5. **Citation compliance:** The evaluation checks for `[Source N]` citations to make grounding visible.

Mitigations used: overlapping chunks, normalized semantic embeddings, Top-K retrieval, low-temperature generation, strict context-only prompting, and explicit source citation formatting.


## 2.7 Export

In [16]:

EVAL_CSV = EXPORT_DIR / "evaluation_results.csv"
evaluation_df.to_csv(EVAL_CSV, index=False)

TEST_JSON = EXPORT_DIR / "test_questions.json"
TEST_JSON.write_text(json.dumps(TEST_QUESTIONS, indent=2), encoding="utf-8")

SOURCE_JSON = EXPORT_DIR / "source_manifest.json"
SOURCE_JSON.write_text(json.dumps(SOURCE_URLS, indent=2), encoding="utf-8")

print("Exported:")
print(" -", EVAL_CSV)
print(" -", TEST_JSON)
print(" -", SOURCE_JSON)
print("\nPersisted vector store:")
print(" -", INDEX_PATH)
print(" -", METADATA_PATH)
print(" -", CONFIG_PATH)


Exported:
 - /content/mars_rag_project/exports/evaluation_results.csv
 - /content/mars_rag_project/exports/test_questions.json
 - /content/mars_rag_project/exports/source_manifest.json

Persisted vector store:
 - /content/mars_rag_project/data/vector_store/mars_faiss.index
 - /content/mars_rag_project/data/vector_store/chunks.json
 - /content/mars_rag_project/data/vector_store/config.json


## Save a Submission Bundle

In [17]:

import shutil

bundle_base = "/content/Mars_RAG_Artifacts"

archive_path = shutil.make_archive(
    bundle_base,
    "zip",
    root_dir=PROJECT_DIR
)

print("Created:", archive_path)

try:
    from google.colab import files
    print("\nTo download the generated data/vector/evaluation ZIP, run:")
    print("files.download(archive_path)")
except Exception:
    pass


Created: /content/Mars_RAG_Artifacts.zip

To download the generated data/vector/evaluation ZIP, run:
files.download(archive_path)



## Optional Interactive Question

Use this cell during your live demo. Change only the `question` string.


In [18]:

question = "What did NASA's InSight mission study on Mars?"
ask_rag(question)


QUESTION:
What did NASA's InSight mission study on Mars?

ANSWER:
According to the retrieved context, NASA's InSight mission studied the interior structure and processes of Mars, including:

* The size of the core, what it is made of, and whether it is liquid or solid.
* The thickness and structure of the crust.
* The structure of the mantle and what it is made of.
* How warm the interior is and how much heat is still flowing through.
* How powerful and frequent internal seismic activity is on Mars, and where it is located within the structure of the planet.
* How often meteorites impact the surface of Mars.

(Source: [Source 1] - insight_science.txt)

RETRIEVED SOURCES:
[Source 1] insight_science.txt | insight_science_chunk_001 | score=0.6894
https://science.nasa.gov/mission/insight/science/
[Source 2] insight.txt | insight_chunk_001 | score=0.6441
https://science.nasa.gov/mission/insight/
[Source 3] insight.txt | insight_chunk_000 | score=0.643
https://science.nasa.gov/mission/insigh

{'question': "What did NASA's InSight mission study on Mars?",
 'answer': "According to the retrieved context, NASA's InSight mission studied the interior structure and processes of Mars, including:\n\n* The size of the core, what it is made of, and whether it is liquid or solid.\n* The thickness and structure of the crust.\n* The structure of the mantle and what it is made of.\n* How warm the interior is and how much heat is still flowing through.\n* How powerful and frequent internal seismic activity is on Mars, and where it is located within the structure of the planet.\n* How often meteorites impact the surface of Mars.\n\n(Source: [Source 1] - insight_science.txt)",
 'sources': [{'source_number': 1,
   'document': 'insight_science.txt',
   'chunk_id': 'insight_science_chunk_001',
   'url': 'https://science.nasa.gov/mission/insight/science/',
   'score': 0.6894},
  {'source_number': 2,
   'document': 'insight.txt',
   'chunk_id': 'insight_chunk_001',
   'url': 'https://science.nasa

In [19]:
from google.colab import files

files.download("/content/Mars_RAG_Artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


# Final Project Summary

### Pipeline
**NASA documents → cleaning → overlapping chunks → MiniLM embeddings → persisted FAISS vector store → Top-K retrieval → grounded prompt → Ollama Llama 3.2 → cited answer**

### Technology Choices
- **NASA Science:** authoritative public source documents
- **Sentence Transformers / MiniLM:** lightweight open-source semantic embeddings
- **FAISS:** fast local vector search with persistence
- **Ollama:** local open-source LLM serving, as required by the project
- **Llama 3.2 3B:** small enough for a practical Colab demonstration
- **Top-K = 4:** gives the LLM multiple relevant context chunks without overloading the prompt
- **Temperature = 0.1:** encourages stable, factual answers
- **Citation-style grounding:** answers identify which retrieved context supported the response

### Submission Check
Before submitting:
1. Run **Runtime → Restart session**, then **Run all**.
2. Confirm every cell finishes without an error.
3. Read the 10 generated answers and manually verify them against the displayed NASA sources.
4. Download `Mars_RAG_Artifacts.zip`.
5. Keep this notebook as `notebooks/rag_pipeline.ipynb` in the final GitHub repository.
